In [ ]:
!nvidia-smi


Sat Oct 17 14:45:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 54.3 MB/s eta 0:00:00


In [ ]:
import torch, time, re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

name = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
tok.padding_side = "left"

2.11.0+cu130 True Tesla T4


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
ids = tok("What is the capital of France?")["input_ids"]
print(ids)
print(tok.convert_ids_to_tokens(ids))
print(tok.decode(ids))

ids = tok("Quantization of transformers is unbelievably useful")["input_ids"]
print(ids)
print(tok.convert_ids_to_tokens(ids))

[3838, 374, 279, 6722, 315, 9625, 30]
['What', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof', 'ĠFrance', '?']
What is the capital of France?
[44220, 2022, 315, 86870, 374, 38937, 88134, 5390]
['Quant', 'ization', 'Ġof', 'Ġtransformers', 'Ġis', 'Ġunbelie', 'vably', 'Ġuseful']


In [ ]:
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
!nvidia-smi

Sat Oct 17 14:46:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             29W /   70W |    3239MiB /  15360MiB |     35%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
def bench(model, device, prompts, max_new_tokens=64, runs=3):
    inputs = tok(prompts, return_tensors="pt", padding=True).to(device)
    gen = dict(max_new_tokens=max_new_tokens, min_new_tokens=max_new_tokens, do_sample=False)
    model.generate(**inputs, **{**gen, "max_new_tokens": 8, "min_new_tokens": 8})  # warmup
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    times = []
    for _ in range(runs):
        if device == "cuda":
            torch.cuda.synchronize()
            s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            s.record()
            out = model.generate(**inputs, **gen)
            e.record(); torch.cuda.synchronize()
            times.append(s.elapsed_time(e) / 1000)
        else:
            t0 = time.perf_counter()
            out = model.generate(**inputs, **gen)
            times.append(time.perf_counter() - t0)
    avg = sum(times) / len(times)
    new_tokens = max_new_tokens * len(prompts)
    mem = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else None
    return {"latency_s": avg, "tokens_per_s": new_tokens / avg, "peak_mem_gb": mem}


In [ ]:
test_prompts = ["What is the capital of France?"]
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")
print(bench(model, "cuda", test_prompts, max_new_tokens=32))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'latency_s': 1.2747003173828124, 'tokens_per_s': 25.103939775979438, 'peak_mem_gb': 3.104342528}


In [ ]:
print(bench(model, "cuda", test_prompts, max_new_tokens=32))
print(bench(model, "cuda", test_prompts, max_new_tokens=32))

{'latency_s': 1.2719462076822916, 'tokens_per_s': 25.158296637646018, 'peak_mem_gb': 3.104342528}
{'latency_s': 1.2660118408203125, 'tokens_per_s': 25.276224888438325, 'peak_mem_gb': 3.104342528}


Baseline, batch size 1, fp16, 32 new tokens, Tesla T4
Session 1: 2.31 s (13.9 tok/s, slow first call), 1.24 s (25.8), 1.26 s (25.4)
Session 2: 1.27 s (25.2), 1.28 s (25.0)
Peak memory: 3.10 GB
Note: the first call in session 1 was ~2x slower and didn't reproduce, probably a one-time cold start.

In [ ]:
prompts = [
    "What is the capital of France?",
    "Explain what a GPU is in two sentences.",
    "Write a Python function that returns the nth Fibonacci number.",
    "List three differences between a CPU and a GPU.",
    "If a train travels 60 miles in 1.5 hours, what is its average speed?",
    "Translate to Spanish: Where is the nearest train station?",
    "Give me three tips for preparing for a technical interview.",
    "What does the word 'latency' mean in computing?",
    "Write a haiku about the ocean.",
    "Explain what quantization means for a neural network in simple terms.",
    "What is 17 multiplied by 23?",
    "Write a short, polite email asking a professor for an extension.",
    "Name three uses of machine learning in healthcare.",
    "What is the difference between a list and a tuple in Python?",
    "Describe how a CUDA thread block works in one paragraph.",
    "Why is the sky blue?",
]
print(len(prompts))

enc = tok(prompts[:4], return_tensors="pt", padding=True)
print(enc["input_ids"].shape)
print(enc["attention_mask"])

16
torch.Size([4, 11])
tensor([[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!nvidia-smi

Sat Oct 17 14:46:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             38W /   70W |     155MiB /  15360MiB |     23%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
cpu_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)  # stays on CPU
print("CPU threads:", torch.get_num_threads())
cpu_result = bench(cpu_model, "cpu", prompts[:1], max_new_tokens=32, runs=2)
print("CPU fp32:", cpu_result)
del cpu_model
gc.collect()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

CPU threads: 4
CPU fp32: {'latency_s': 5.733960985500005, 'tokens_per_s': 5.580784396845627, 'peak_mem_gb': None}


2747

In [ ]:
gpu_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to("cuda")
gpu_result = bench(gpu_model, "cuda", prompts[:1], max_new_tokens=32, runs=3)
print("GPU fp32:", gpu_result)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

GPU fp32: {'latency_s': 1.3434816487630208, 'tokens_per_s': 23.818710162109955, 'peak_mem_gb': 6.192115712}


In [ ]:
print("GPU fp32 vs CPU fp32 speedup:", gpu_result["tokens_per_s"] / cpu_result["tokens_per_s"])

GPU fp32 vs CPU fp32 speedup: 4.267986087327218


In [ ]:
del gpu_model
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to("cuda")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
from torch.profiler import profile, ProfilerActivity

inputs = tok(prompts[:1], return_tensors="pt").to("cuda")

for _ in range(2):
    model.generate(**inputs, max_new_tokens=8, do_sample=False)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    model.generate(**inputs, max_new_tokens=16, do_sample=False)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::linear         1.88%      15.477ms        22.52%     185.719ms      58.921us       0.000us         0.00%     214.585ms      68.079us          3152  
                                           aten::matmul         2.58%      21.316ms        17.81%     146.897ms      54.006us       0.000us         0.00%     201.157ms      73.955us          2720  
         

In [ ]:
results = []
for bs in [1, 4, 16]:
    r = bench(model, "cuda", prompts[:bs], max_new_tokens=32, runs=3)
    r["batch"] = bs
    results.append(r)
    print(bs, r)

1 {'latency_s': 1.5494956868489584, 'tokens_per_s': 20.651880654843858, 'peak_mem_gb': 3.104343552, 'batch': 1}
4 {'latency_s': 1.4714621175130207, 'tokens_per_s': 86.9883080757377, 'peak_mem_gb': 3.119308288, 'batch': 4}
16 {'latency_s': 1.5037613118489583, 'tokens_per_s': 340.47956678075957, 'peak_mem_gb': 3.176977408, 'batch': 16}


Batch	Latency (s)	Tokens/s	vs batch 1	Peak mem (GB)
1	1.62	19.8	1x	3.104
4	1.50	85.5	4.3x	3.119
16	1.56	329.1	16.6x	3.177

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print("still defined:", [n for n in ["model", "gpu_model", "cpu_model"] if n in globals()])
print("missing setup:", [n for n in ["tok", "bench", "prompts", "name"] if n not in globals()])

still defined: []
missing setup: []


In [ ]:
for n in ["model", "gpu_model", "cpu_model", "prof", "inputs"]:
    if n in globals():
        del globals()[n]
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!nvidia-smi

Sat Oct 17 14:50:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             31W /   70W |     169MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
def load(mode):
    if mode == "fp32":
        return AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to("cuda")
    if mode == "fp16":
        return AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to("cuda")
    if mode == "int8":
        cfg = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(name, quantization_config=cfg, device_map="cuda")
    if mode == "int4":
        cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        return AutoModelForCausalLM.from_pretrained(name, quantization_config=cfg, device_map="cuda")

In [ ]:
import pandas as pd

rows = []
for mode in ["fp32", "fp16", "int8", "int4"]:
    m = load(mode)
    for bs in [1, 4, 16]:
        r = bench(m, "cuda", prompts[:bs], max_new_tokens=32, runs=3)
        r.update({"mode": mode, "batch": bs})
        rows.append(r)
        print(mode, bs, round(r["tokens_per_s"], 1), "tok/s,", round(r["peak_mem_gb"], 2), "GB")
    del m
    gc.collect()
    torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv("precision_results.csv", index=False)
df

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

fp32 1 22.6 tok/s, 6.19 GB
fp32 4 86.8 tok/s, 6.21 GB
fp32 16 242.8 tok/s, 6.29 GB


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

fp16 1 20.7 tok/s, 3.1 GB
fp16 4 87.9 tok/s, 3.12 GB
fp16 16 339.3 tok/s, 3.18 GB


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

Streaming output truncated to the last 5000 lines.


int8 16 34.1 tok/s, 1.87 GB


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

int4 1 11.9 tok/s, 1.23 GB
int4 4 42.8 tok/s, 1.24 GB
int4 16 170.3 tok/s, 1.3 GB


,latency_s,tokens_per_s,peak_mem_gb,mode,batch
0,1.416279,22.594423,6.192116,fp32,1
1,1.474899,86.785574,6.210517,fp32,4
2,2.108633,242.811378,6.292385,fp32,16
3,1.544578,20.717631,3.104343,fp16,1
4,1.455699,87.930282,3.119307,fp16,4
5,1.508858,339.329509,3.176976,fp16,16
6,14.354286,2.229299,1.801260,int8,1
7,15.024841,8.519225,1.815781,int8,4
8,15.020993,34.085628,1.874222,int8,16
9,2.678035,11.949061,1.229658,int4,1


In [42]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for mode, g in df.groupby("mode", sort=False):
    ax[0].plot(g["batch"], g["tokens_per_s"], marker="o", label=mode)
ax[0].set_xlabel("batch size"); ax[0].set_ylabel("tokens / second")
ax[0].set_title("Throughput vs batch size"); ax[0].legend()

mem = df.groupby("mode", sort=False)["peak_mem_gb"].max()
ax[1].bar(mem.index, mem.values)
ax[1].set_ylabel("peak GPU memory (GB)"); ax[1].set_title("Memory by precision")
plt.tight_layout(); plt.savefig("precision_charts.png", dpi=150)
print("saved")

saved


In [43]:
from numba import cuda
import numpy as np, math

@cuda.jit
def vec_add(a, b, c):
    i = cuda.grid(1)      # this thread's global index
    if i < c.size:        # extra threads from rounding up must not write out of bounds
        c[i] = a[i] + b[i]

n = 10_000_000
a = np.random.rand(n).astype(np.float32)
b = np.random.rand(n).astype(np.float32)

d_a, d_b = cuda.to_device(a), cuda.to_device(b)    # CPU -> GPU memory
d_c = cuda.device_array(n, dtype=np.float32)       # empty output on the GPU

threads = 256
blocks = math.ceil(n / threads)                    # round up to cover all n
vec_add[blocks, threads](d_a, d_b, d_c)
cuda.synchronize()                                 # launch is async, so wait

print(np.allclose(d_c.copy_to_host(), a + b))

True


In [46]:
from numba import cuda
import numpy as np, math, time, torch

@cuda.jit
def matmul_naive(A, B, C):
    col, row = cuda.grid(2)           # x = column, y = row; x varies fastest across threads
    if row < C.shape[0] and col < C.shape[1]:
        s = 0.0
        for k in range(A.shape[1]):   # dot product of one row of A and one column of B
            s += A[row, k] * B[k, col]
        C[row, col] = s

N = 1024
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)

dA, dB = cuda.to_device(A), cuda.to_device(B)
dC = cuda.device_array((N, N), dtype=np.float32)

tpb = (16, 16)                                    # 256 threads per block
bpg = (math.ceil(N / 16), math.ceil(N / 16))      # blocks per grid, 2D

matmul_naive[bpg, tpb](dA, dB, dC)                # warmup: first call includes JIT compile time
cuda.synchronize()

# time my kernel
t0 = time.perf_counter()
for _ in range(10):
    matmul_naive[bpg, tpb](dA, dB, dC)
cuda.synchronize()                                # wait before stopping the clock
mine_ms = (time.perf_counter() - t0) / 10 * 1000

# time PyTorch's matmul (cuBLAS underneath) on the same data
tA, tB = torch.from_numpy(A).cuda(), torch.from_numpy(B).cuda()
torch.matmul(tA, tB)                              # warmup
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    tC = torch.matmul(tA, tB)
torch.cuda.synchronize()
torch_ms = (time.perf_counter() - t0) / 10 * 1000

# correctness first, then speed
print("correct:", np.allclose(dC.copy_to_host(), tC.cpu().numpy(), rtol=1e-3, atol=1e-3))
print(f"my kernel: {mine_ms:.2f} ms | torch/cuBLAS: {torch_ms:.2f} ms | ratio: {mine_ms / torch_ms:.1f}x slower")

correct: True
my kernel: 33.23 ms | torch/cuBLAS: 0.38 ms | ratio: 87.1x slower


In [47]:
from numba import cuda, float32

TPB = 16   # tile size = threads per block side

@cuda.jit
def matmul_tiled(A, B, C):
    sA = cuda.shared.array((TPB, TPB), dtype=float32)   # fast memory shared by the block
    sB = cuda.shared.array((TPB, TPB), dtype=float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    col, row = cuda.grid(2)
    acc = 0.0
    for t in range((A.shape[1] + TPB - 1) // TPB):      # walk across the tiles
        # each thread loads one element of each tile (0 if out of bounds)
        sA[ty, tx] = A[row, t*TPB + tx] if row < A.shape[0] and t*TPB + tx < A.shape[1] else 0.0
        sB[ty, tx] = B[t*TPB + ty, col] if col < B.shape[1] and t*TPB + ty < B.shape[0] else 0.0
        cuda.syncthreads()                              # wait until the whole tile is loaded
        for k in range(TPB):
            acc += sA[ty, k] * sB[k, tx]
        cuda.syncthreads()                              # wait before the tile is overwritten
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = acc

# same launch shape as before; warmup, then time it
dC2 = cuda.device_array((N, N), dtype=np.float32)
matmul_tiled[bpg, tpb](dA, dB, dC2); cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    matmul_tiled[bpg, tpb](dA, dB, dC2)
cuda.synchronize()
tiled_ms = (time.perf_counter() - t0) / 10 * 1000

print("correct:", np.allclose(dC2.copy_to_host(), tC.cpu().numpy(), rtol=1e-3, atol=1e-3))
print(f"tiled: {tiled_ms:.2f} ms | vs naive: {mine_ms / tiled_ms:.1f}x faster | vs cuBLAS: {tiled_ms / torch_ms:.1f}x slower")

correct: True
tiled: 36.01 ms | vs naive: 0.9x faster | vs cuBLAS: 94.4x slower


In [50]:
def time_kernel(k, out, reps=20):
    k[bpg, tpb](dA, dB, out); cuda.synchronize()    # warmup
    t0 = time.perf_counter()
    for _ in range(reps):
        k[bpg, tpb](dA, dB, out)
    cuda.synchronize()
    return (time.perf_counter() - t0) / reps * 1000

for run in range(6):
    print(f"naive {time_kernel(matmul_naive, dC):.2f} ms | tiled {time_kernel(matmul_tiled, dC2):.2f} ms")

naive 28.03 ms | tiled 21.68 ms
naive 21.88 ms | tiled 21.69 ms
naive 21.88 ms | tiled 21.72 ms
naive 22.02 ms | tiled 21.70 ms
naive 22.10 ms | tiled 21.72 ms
naive 22.08 ms | tiled 21.72 ms


In [52]:
!nvidia-smi -q -d CLOCK,TEMPERATURE


==============NVSMI LOG==============

Timestamp                                 : Sat Oct 17 16:37:29 2026
Driver Version                            : 580.82.07
CUDA Version                              : 13.0

Attached GPUs                             : 1
GPU 00000000:00:04.0
    Temperature
        GPU Current Temp                  : 66 C
        GPU T.Limit Temp                  : N/A
        GPU Shutdown Temp                 : 96 C
        GPU Slowdown Temp                 : 93 C
        GPU Max Operating Temp            : 85 C
        GPU Target Temperature            : N/A
        Memory Current Temp               : N/A
        Memory Max Operating Temp         : N/A
    Clocks
        Graphics                          : 585 MHz
        SM                                : 585 MHz
        Memory                            : 5000 MHz
        Video                             : 735 MHz
    Applications Clocks
        Graphics                          : 585 MHz
        Memory      

| Kernel | Time | vs cuBLAS |
| :--- | :--- | :--- |
| **Naive** | ~22.9 ms | ~28x slower |
| **Tiled (16x16)** | ~21.9 ms | ~26x slower |
| **cuBLAS** | ~0.83 ms | 1x |